# 05A — Static Feature Extraction

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

## Inputs

This notebook uses:

- `malaysia_positive_inventory_ml_eligible.geojson`
- `background_candidates_3000.geojson`

The current workflow contains approximately:

- **200 point-ML-eligible landslide positives**
- **3,000 background candidates**

The exact counts are always read from the files rather than hard-coded.

---

# Features extracted in Stage 05A

## Terrain — Google Earth Engine / NASA SRTM

- elevation
- slope
- aspect

## Soil — local SoilGrids rasters

- clay
- sand
- silt
- bulk density

## Geology — JMG MyGEMS API

- lithology / geological unit attributes
- source JMG layer

## Faulting — JMG MyGEMS API

- distance to nearest major fault

---

# Not included yet

These are intentionally postponed to **Stage 05B** because their values may depend on event/reference time:

- historical land cover
- historical/current road distance
- waterways
- rainfall

---

# Candidate-validity rule

The environmental factors are **not used to choose only low-risk pseudo-absences**.

A background candidate may later be rejected only for objective data-validity reasons such as:

- missing DEM
- NoData soil raster
- outside valid land mask
- missing required predictor coverage

A candidate is **not** rejected merely because it has:

- steep slope
- high elevation
- proximity to faults
- a particular geology type

Those are predictors the models should learn from.


In [1]:
# OPTIONAL — install once if needed
# %pip install earthengine-api geopandas rasterio requests pyproj shapely pandas numpy matplotlib

from pathlib import Path
import json
import math
import re
import time
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import requests
import ee

from shapely.geometry import shape
from shapely.ops import nearest_points

warnings.filterwarnings("ignore", category=UserWarning)

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("rasterio:", rasterio.__version__)


pandas: 3.0.2
geopandas: 1.1.4
rasterio: 1.5.0


## 1. Project paths

Change only:

- `PROJECT_ROOT`
- `EE_PROJECT`

Your current folders already match this notebook.


In [2]:
# ============================================================
# CHANGE THESE TWO VALUES
# ============================================================

PROJECT_ROOT = Path(r"C:\Landslide_FYP")

# Google Cloud / Earth Engine project already tested in earthengine.ipynb
EE_PROJECT = "fyp-landslide-malaysia"


# ============================================================
# PROJECT STRUCTURE
# ============================================================

RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

POSITIVE_PATH = (
    DEV_DIR
    / "02_cleaned_inventory"
    / "malaysia_positive_inventory_ml_eligible.geojson"
)

BACKGROUND_PATH = (
    DEV_DIR
    / "02_cleaned_inventory"
    / "background_candidates_3000.geojson"
)

SOIL_ROOT = (
    RAW_DIR
    / "03_Factors"
    / "Soil"
)

SOIL_PATHS = {
    "soil_clay": (
        SOIL_ROOT
        / "Clay"
        / "soilgrids_clay_malaysia.tif"
    ),
    "soil_sand": (
        SOIL_ROOT
        / "Sand"
        / "soilgrids_sand_malaysia.tif"
    ),
    "soil_silt": (
        SOIL_ROOT
        / "Silt"
        / "soilgrids_silt_malaysia.tif"
    ),
    "soil_bulk_density": (
        SOIL_ROOT
        / "Bulk_Density"
        / "soilgrids_bdod_malaysia.tif"
    ),
}

OUTPUT_DIR = (
    DEV_DIR
    / "03_derived_features"
    / "static_features"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# INPUT CHECKS
# ============================================================

required_paths = {
    "ML-eligible positives": POSITIVE_PATH,
    "Background candidates": BACKGROUND_PATH,
    **SOIL_PATHS
}

for name, path in required_paths.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found:\\n{path}"
        )

    print("FOUND:", name, "->", path)

print("\\nOutput folder:")
print(OUTPUT_DIR)


FOUND: ML-eligible positives -> C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_ml_eligible.geojson
FOUND: Background candidates -> C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\background_candidates_3000.geojson
FOUND: soil_clay -> C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Soil\Clay\soilgrids_clay_malaysia.tif
FOUND: soil_sand -> C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Soil\Sand\soilgrids_sand_malaysia.tif
FOUND: soil_silt -> C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Soil\Silt\soilgrids_silt_malaysia.tif
FOUND: soil_bulk_density -> C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Soil\Bulk_Density\soilgrids_bdod_malaysia.tif
\nOutput folder:
C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features


## 2. Load positives and background candidates

At this stage:

- positives → `target = 1`
- background candidates → `target` remains missing

The candidates are **not yet final pseudo-absences**.


In [3]:
positive = (
    gpd.read_file(POSITIVE_PATH)
    .to_crs("EPSG:4326")
)

background = (
    gpd.read_file(BACKGROUND_PATH)
    .to_crs("EPSG:4326")
)


# ------------------------------------------------------------
# Standard sample IDs
# ------------------------------------------------------------

positive = positive.copy()

positive["sample_id"] = (
    positive["final_positive_id"]
    .astype("string")
)

positive["sample_class"] = "landslide"
positive["target"] = 1


background = background.copy()

background["sample_id"] = (
    background["background_id"]
    .astype("string")
)

background["sample_class"] = "background_candidate"
background["target"] = pd.NA


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

keep_common = [
    "sample_id",
    "sample_class",
    "target",
    "geometry",
]

samples = gpd.GeoDataFrame(
    pd.concat(
        [
            positive[keep_common],
            background[keep_common]
        ],
        ignore_index=True
    ),
    geometry="geometry",
    crs="EPSG:4326"
)

samples["longitude"] = samples.geometry.x
samples["latitude"] = samples.geometry.y


print("Positive samples:", len(positive))
print("Background candidates:", len(background))
print("Combined:", len(samples))

display(
    samples["sample_class"]
    .value_counts()
    .rename_axis("sample_class")
    .reset_index(name="records")
)


Positive samples: 200
Background candidates: 3000
Combined: 3200


,sample_class,records
0,background_candidate,3000
1,landslide,200


# PART A — TERRAIN

## 3. Connect to Google Earth Engine

This uses the same Earth Engine project that you already verified.

If authentication is requested, complete the browser login once.


In [4]:
try:

    ee.Initialize(
        project=EE_PROJECT
    )

    print(
        "Earth Engine already authenticated "
        "and initialized."
    )

except Exception:

    print(
        "Earth Engine authentication required..."
    )

    ee.Authenticate()

    ee.Initialize(
        project=EE_PROJECT
    )

    print(
        "Earth Engine authenticated and initialized."
    )


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine already authenticated and initialized.


## 4. Build SRTM elevation, slope and aspect image

Dataset:

`USGS/SRTMGL1_003`

Slope and aspect are derived using `ee.Terrain`.


In [5]:
srtm = (
    ee.Image("USGS/SRTMGL1_003")
    .select("elevation")
)

terrain = ee.Terrain.products(srtm)

terrain_image = (
    ee.Image.cat([
        srtm.rename("elevation_m"),
        terrain.select("slope").rename("slope_deg"),
        terrain.select("aspect").rename("aspect_deg"),
    ])
)

print(
    "Terrain image prepared:",
    terrain_image.bandNames().getInfo()
)


Terrain image prepared: ['elevation_m', 'slope_deg', 'aspect_deg']


## 5. Extract terrain features in chunks

The notebook uses small chunks rather than making one very large Earth Engine request.

This is slower than one huge query but much easier to recover if the connection temporarily fails.


In [6]:
TERRAIN_CHUNK_SIZE = 250
TERRAIN_SCALE_M = 30


def extract_terrain_chunk(
    chunk_df,
    image,
    scale=30
):

    features = []

    for _, row in chunk_df.iterrows():

        geom = ee.Geometry.Point([
            float(row["longitude"]),
            float(row["latitude"])
        ])

        features.append(
            ee.Feature(
                geom,
                {
                    "sample_id":
                        str(row["sample_id"])
                }
            )
        )

    fc = ee.FeatureCollection(features)

    sampled = image.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.first(),
        scale=scale
    )

    result = sampled.getInfo()

    rows = []

    for feat in result.get(
        "features",
        []
    ):

        props = feat.get(
            "properties",
            {}
        )

        rows.append({
            "sample_id":
                props.get("sample_id"),

            "elevation_m":
                props.get("elevation_m"),

            "slope_deg":
                props.get("slope_deg"),

            "aspect_deg":
                props.get("aspect_deg")
        })

    return pd.DataFrame(rows)


terrain_parts = []

for start in range(
    0,
    len(samples),
    TERRAIN_CHUNK_SIZE
):

    stop = min(
        start + TERRAIN_CHUNK_SIZE,
        len(samples)
    )

    print(
        f"Terrain extraction "
        f"{start + 1}-{stop} "
        f"of {len(samples)}"
    )

    chunk = samples.iloc[
        start:stop
    ]

    part = extract_terrain_chunk(
        chunk,
        terrain_image,
        TERRAIN_SCALE_M
    )

    terrain_parts.append(part)

    # Avoid rapid repeated API calls
    time.sleep(0.5)


terrain_features = pd.concat(
    terrain_parts,
    ignore_index=True
)


print(
    "Terrain records returned:",
    len(terrain_features)
)

display(
    terrain_features.head()
)


Terrain extraction 1-250 of 3200
Terrain extraction 251-500 of 3200
Terrain extraction 501-750 of 3200
Terrain extraction 751-1000 of 3200
Terrain extraction 1001-1250 of 3200
Terrain extraction 1251-1500 of 3200
Terrain extraction 1501-1750 of 3200
Terrain extraction 1751-2000 of 3200
Terrain extraction 2001-2250 of 3200
Terrain extraction 2251-2500 of 3200
Terrain extraction 2501-2750 of 3200
Terrain extraction 2751-3000 of 3200
Terrain extraction 3001-3200 of 3200
Terrain records returned: 3200


,sample_id,elevation_m,slope_deg,aspect_deg
0,POS_0001,148,39,304
1,POS_0002,605,20,157
2,POS_0003,1222,6,90
3,POS_0006,166,18,317
4,POS_0007,48,10,343


In [7]:
# Merge terrain values back to samples

samples = samples.merge(
    terrain_features,
    on="sample_id",
    how="left"
)

samples = gpd.GeoDataFrame(
    samples,
    geometry="geometry",
    crs="EPSG:4326"
)


samples["terrain_valid"] = (
    samples[
        [
            "elevation_m",
            "slope_deg",
            "aspect_deg"
        ]
    ]
    .notna()
    .all(axis=1)
)


print(
    "Valid terrain:",
    int(samples["terrain_valid"].sum()),
    "/",
    len(samples)
)

display(
    samples[
        [
            "sample_id",
            "elevation_m",
            "slope_deg",
            "aspect_deg",
            "terrain_valid"
        ]
    ].head()
)


Valid terrain: 3200 / 3200


,sample_id,elevation_m,slope_deg,aspect_deg,terrain_valid
0,POS_0001,148,39,304,True
1,POS_0002,605,20,157,True
2,POS_0003,1222,6,90,True
3,POS_0006,166,18,317,True
4,POS_0007,48,10,343,True


# PART B — SOIL

## 6. Raster sampling helper

Each SoilGrids raster is sampled at the exact point after transforming the WGS84 point into the raster CRS.

No interpolation or missing-value filling is performed.

The original raster values are preserved.

**Important:** SoilGrids products may use scale factors depending on the downloaded product. This notebook records raw raster values first. Units/scaling should be confirmed before final modelling.


In [8]:
from pyproj import Transformer


def sample_raster_at_points(
    raster_path,
    points_gdf
):

    values = []

    with rasterio.open(
        raster_path
    ) as src:

        print(
            "\nRaster:",
            raster_path.name
        )

        print(
            "CRS:",
            src.crs
        )

        print(
            "Resolution:",
            src.res
        )

        print(
            "NoData:",
            src.nodata
        )

        transformer = Transformer.from_crs(
            "EPSG:4326",
            src.crs,
            always_xy=True
        )

        coords = []

        for geom in points_gdf.geometry:

            x, y = transformer.transform(
                geom.x,
                geom.y
            )

            coords.append(
                (x, y)
            )

        sampled = list(
            src.sample(coords)
        )

        for array in sampled:

            value = (
                array[0]
                if len(array)
                else np.nan
            )

            if (
                src.nodata is not None
                and value == src.nodata
            ):
                value = np.nan

            values.append(value)

    return values


## 7. Extract all four SoilGrids variables


In [9]:
soil_metadata_rows = []

for feature_name, raster_path in SOIL_PATHS.items():

    print(
        "\nExtracting:",
        feature_name
    )

    samples[feature_name] = (
        sample_raster_at_points(
            raster_path,
            samples
        )
    )

    with rasterio.open(
        raster_path
    ) as src:

        soil_metadata_rows.append({
            "feature":
                feature_name,

            "file":
                raster_path.name,

            "crs":
                str(src.crs),

            "resolution_x":
                src.res[0],

            "resolution_y":
                src.res[1],

            "nodata":
                src.nodata,

            "dtype":
                str(src.dtypes[0]),

            "min_sample":
                pd.to_numeric(
                    samples[feature_name],
                    errors="coerce"
                ).min(),

            "max_sample":
                pd.to_numeric(
                    samples[feature_name],
                    errors="coerce"
                ).max(),

            "missing_samples":
                int(
                    samples[
                        feature_name
                    ].isna().sum()
                ),
        })


soil_metadata = pd.DataFrame(
    soil_metadata_rows
)

display(soil_metadata)

soil_metadata.to_csv(
    OUTPUT_DIR
    / "soil_raster_metadata_and_sample_summary.csv",
    index=False
)



Extracting: soil_clay

Raster: soilgrids_clay_malaysia.tif
CRS: ESRI:54009
Resolution: (250.0, 250.0)
NoData: None

Extracting: soil_sand

Raster: soilgrids_sand_malaysia.tif
CRS: ESRI:54009
Resolution: (250.0, 250.0)
NoData: None

Extracting: soil_silt

Raster: soilgrids_silt_malaysia.tif
CRS: ESRI:54009
Resolution: (250.0, 250.0)
NoData: None

Extracting: soil_bulk_density

Raster: soilgrids_bdod_malaysia.tif
CRS: ESRI:54009
Resolution: (250.0, 250.0)
NoData: None


,feature,file,crs,resolution_x,resolution_y,nodata,dtype,min_sample,max_sample,missing_samples
0,soil_clay,soilgrids_clay_malaysia.tif,ESRI:54009,250.0,250.0,None,int16,0,669,0
1,soil_sand,soilgrids_sand_malaysia.tif,ESRI:54009,250.0,250.0,None,int16,0,527,0
2,soil_silt,soilgrids_silt_malaysia.tif,ESRI:54009,250.0,250.0,None,int16,0,438,0
3,soil_bulk_density,soilgrids_bdod_malaysia.tif,ESRI:54009,250.0,250.0,None,int16,0,132,0


In [10]:
soil_cols = [
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
]

samples["soil_complete"] = (
    samples[soil_cols]
    .notna()
    .all(axis=1)
)

print(
    "Complete soil coverage:",
    int(samples["soil_complete"].sum()),
    "/",
    len(samples)
)


Complete soil coverage: 3200 / 3200


# PART C — JMG LITHOLOGY

## 8. JMG API configuration

The MyGEMS service already verified in your test notebook is:

`Demarcation/Litology_by_Negeri/FeatureServer`

This notebook downloads the state polygon layers into memory and performs the point-in-polygon join locally.

The raw API result is also cached into the Stage-05A output folder so repeated runs do not need to redownload everything.


In [11]:
JMG_LITHOLOGY_SERVICE = (
    "https://mygems.jmg.gov.my/"
    "server/rest/services/"
    "Demarcation/"
    "Litology_by_Negeri/"
    "FeatureServer"
)

LITH_CACHE = (
    OUTPUT_DIR
    / "jmg_lithology_all_layers.geojson"
)

session = requests.Session()


def arcgis_json(
    url,
    params=None,
    timeout=90
):

    params = {
        "f": "json",
        **(params or {})
    }

    response = session.get(
        url,
        params=params,
        timeout=timeout
    )

    response.raise_for_status()

    result = response.json()

    if "error" in result:
        raise RuntimeError(
            result["error"]
        )

    return result


service_meta = arcgis_json(
    JMG_LITHOLOGY_SERVICE
)

layers_meta = (
    service_meta.get(
        "layers",
        []
    )
)

print(
    "Lithology layers available:",
    len(layers_meta)
)

display(
    pd.DataFrame(
        layers_meta
    )[
        [
            "id",
            "name"
        ]
    ]
)


Lithology layers available: 30


,id,name
0,0,Lithology of Perlis
1,1,Rock age of Perlis
2,2,Lithology of Kedah
3,3,Rock age of Kedah
4,4,Lithology of Pulau_Pinang
5,5,Rock age of Pulau_Pinang
6,6,Lithology of Perak
7,7,Rock age of Perak
8,8,Lithology of Kelantan
9,9,Rock age of Kelantan


## 9. Download all lithology polygons

ArcGIS FeatureServer queries are paginated.

The function requests GeoJSON directly and preserves all returned attributes.


In [12]:
def download_arcgis_layer_geojson(
    layer_url,
    layer_id,
    layer_name,
    batch_size=1000
):

    offset = 0
    all_features = []

    while True:

        params = {
            "where":
                "1=1",

            "outFields":
                "*",

            "returnGeometry":
                "true",

            "outSR":
                4326,

            "resultOffset":
                offset,

            "resultRecordCount":
                batch_size,

            "f":
                "geojson"
        }

        response = session.get(
            f"{layer_url}/query",
            params=params,
            timeout=120
        )

        response.raise_for_status()

        data = response.json()

        if "error" in data:
            raise RuntimeError(
                data["error"]
            )

        features = data.get(
            "features",
            []
        )

        if not features:
            break

        for feat in features:

            feat.setdefault(
                "properties",
                {}
            )

            feat["properties"][
                "_jmg_layer_id"
            ] = layer_id

            feat["properties"][
                "_jmg_layer_name"
            ] = layer_name

        all_features.extend(
            features
        )

        print(
            layer_name,
            "downloaded:",
            len(all_features)
        )

        if len(features) < batch_size:
            break

        offset += batch_size

        time.sleep(0.25)

    if not all_features:
        return None

    return gpd.GeoDataFrame.from_features(
        all_features,
        crs="EPSG:4326"
    )


In [13]:
if LITH_CACHE.exists():

    print(
        "Using cached lithology file:"
    )

    print(LITH_CACHE)

    lithology = (
        gpd.read_file(
            LITH_CACHE
        )
        .to_crs("EPSG:4326")
    )

else:

    lith_parts = []

    for layer in layers_meta:

        layer_id = layer["id"]
        layer_name = layer["name"]

        print(
            "\nDownloading:",
            layer_id,
            layer_name
        )

        gdf = download_arcgis_layer_geojson(
            f"{JMG_LITHOLOGY_SERVICE}/{layer_id}",
            layer_id,
            layer_name
        )

        if (
            gdf is not None
            and len(gdf) > 0
        ):

            lith_parts.append(gdf)


    if not lith_parts:

        raise RuntimeError(
            "No lithology polygons were returned "
            "from the JMG service."
        )


    lithology = gpd.GeoDataFrame(
        pd.concat(
            lith_parts,
            ignore_index=True,
            sort=False
        ),
        geometry="geometry",
        crs="EPSG:4326"
    )


    lithology.to_file(
        LITH_CACHE,
        driver="GeoJSON"
    )


print(
    "Total lithology polygons:",
    len(lithology)
)

print(
    "Columns:"
)

print(
    lithology.columns.tolist()
)



Downloading: 0 Lithology of Perlis
Lithology of Perlis downloaded: 78

Downloading: 1 Rock age of Perlis
Rock age of Perlis downloaded: 78

Downloading: 2 Lithology of Kedah
Lithology of Kedah downloaded: 902

Downloading: 3 Rock age of Kedah
Rock age of Kedah downloaded: 902

Downloading: 4 Lithology of Pulau_Pinang
Lithology of Pulau_Pinang downloaded: 82

Downloading: 5 Rock age of Pulau_Pinang
Rock age of Pulau_Pinang downloaded: 82

Downloading: 6 Lithology of Perak
Lithology of Perak downloaded: 782

Downloading: 7 Rock age of Perak
Rock age of Perak downloaded: 782

Downloading: 8 Lithology of Kelantan
Lithology of Kelantan downloaded: 780

Downloading: 9 Rock age of Kelantan
Rock age of Kelantan downloaded: 780

Downloading: 10 Lithology of Terengganu
Lithology of Terengganu downloaded: 826

Downloading: 11 Rock age of Terengganu
Rock age of Terengganu downloaded: 826

Downloading: 12 Lithology of Pahang
Lithology of Pahang downloaded: 1000
Lithology of Pahang downloaded: 1510

## 10. Identify likely lithology-description fields

Because JMG state layers may not use exactly the same attribute names, this notebook searches for likely geology/lithology columns.

It does **not invent a class** if none is available.


In [14]:
attribute_cols = [
    c
    for c in lithology.columns
    if c != "geometry"
]


LIKELY_PATTERNS = [
    "lith",
    "lito",
    "rock",
    "batuan",
    "geol",
    "formation",
    "formasi",
    "unit",
    "age",
    "usia",
]


likely_lithology_cols = []

for col in attribute_cols:

    low = col.lower()

    if any(
        pattern in low
        for pattern in LIKELY_PATTERNS
    ):

        likely_lithology_cols.append(
            col
        )


print(
    "Possible lithology/geology fields:"
)

print(
    likely_lithology_cols
)


# Save field inventory for audit
field_inventory = pd.DataFrame({
    "column":
        attribute_cols,

    "likely_lithology_related":
        [
            c in likely_lithology_cols
            for c in attribute_cols
        ]
})

field_inventory.to_csv(
    OUTPUT_DIR
    / "jmg_lithology_field_inventory.csv",
    index=False
)


Possible lithology/geology fields:
['AGECODE', 'AGE']


## 11. Spatially join sample points to lithology polygons

The notebook keeps:

- JMG source layer name
- JMG layer ID
- one automatically constructed `lithology_raw` text field

The original downloaded polygon layer is cached for later manual attribute review.


In [15]:
# ------------------------------------------------------------
# Build a single descriptive lithology field
# using the first non-null likely attribute.
# ------------------------------------------------------------

def first_nonempty_value(
    row,
    columns
):

    for col in columns:

        value = row.get(
            col
        )

        if pd.notna(value):

            text = str(
                value
            ).strip()

            if (
                text
                and text.lower()
                not in {
                    "nan",
                    "none",
                    "null"
                }
            ):
                return text

    return pd.NA


if likely_lithology_cols:

    lithology[
        "lithology_raw"
    ] = lithology.apply(
        first_nonempty_value,
        axis=1,
        columns=likely_lithology_cols
    )

else:

    lithology[
        "lithology_raw"
    ] = pd.NA


lith_join = gpd.sjoin(
    samples[
        [
            "sample_id",
            "geometry"
        ]
    ],
    lithology[
        [
            "_jmg_layer_id",
            "_jmg_layer_name",
            "lithology_raw",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)


# If overlapping polygons produce duplicates,
# preserve the first match but record count.
match_counts = (
    lith_join
    .groupby("sample_id")
    .size()
    .rename(
        "lithology_polygon_matches"
    )
)


lith_first = (
    lith_join
    .sort_values(
        [
            "sample_id",
            "_jmg_layer_id"
        ],
        na_position="last"
    )
    .drop_duplicates(
        "sample_id",
        keep="first"
    )
    [
        [
            "sample_id",
            "_jmg_layer_id",
            "_jmg_layer_name",
            "lithology_raw"
        ]
    ]
)


samples = samples.merge(
    lith_first,
    on="sample_id",
    how="left"
)


samples[
    "lithology_polygon_matches"
] = (
    samples["sample_id"]
    .map(match_counts)
    .fillna(0)
    .astype(int)
)


samples[
    "lithology_available"
] = (
    samples[
        "_jmg_layer_id"
    ].notna()
)


print(
    "Samples with lithology:",
    int(
        samples[
            "lithology_available"
        ].sum()
    ),
    "/",
    len(samples)
)


display(
    samples[
        [
            "sample_id",
            "_jmg_layer_name",
            "lithology_raw",
            "lithology_polygon_matches",
            "lithology_available"
        ]
    ].head(20)
)


Samples with lithology: 3166 / 3200


,sample_id,_jmg_layer_name,lithology_raw,lithology_polygon_matches,lithology_available
0,POS_0001,Lithology of Pulau_Pinang,9500.0,2,True
1,POS_0002,Lithology of Pulau_Pinang,9500.0,2,True
2,POS_0003,Lithology of Sabah,NaN,2,True
3,POS_0006,Lithology of Pahang,9500.0,2,True
4,POS_0007,Lithology of Kuala Lumpur,9500.0,2,True
5,POS_0008,Lithology of Perak,1000.0,2,True
6,POS_0009,Lithology of Pulau_Pinang,9500.0,2,True
7,POS_0010,Lithology of Perak,8000.0,2,True
8,POS_0013,Lithology of Selangor,9500.0,2,True
9,POS_0014,Lithology of Perak,1000.0,2,True


# PART D — MAJOR FAULT DISTANCE

## 12. Download JMG major-fault layer

The verified endpoint is layer `/5`.

The downloaded layer is cached locally in the Stage-05A output folder.


In [16]:
JMG_FAULT_LAYER = (
    "https://mygems.jmg.gov.my/"
    "server/rest/services/"
    "GeologiAsas/"
    "Major_Fault/"
    "FeatureServer/5"
)

FAULT_CACHE = (
    OUTPUT_DIR
    / "jmg_major_fault_layer5.geojson"
)


if FAULT_CACHE.exists():

    print(
        "Using cached major fault layer."
    )

    faults = (
        gpd.read_file(
            FAULT_CACHE
        )
        .to_crs("EPSG:4326")
    )

else:

    faults = download_arcgis_layer_geojson(
        JMG_FAULT_LAYER,
        layer_id=5,
        layer_name="Major_Fault"
    )

    if (
        faults is None
        or len(faults) == 0
    ):

        raise RuntimeError(
            "JMG major-fault API returned no features."
        )

    faults.to_file(
        FAULT_CACHE,
        driver="GeoJSON"
    )


print(
    "Major fault features:",
    len(faults)
)

print(
    "Geometry types:",
    faults.geom_type
    .value_counts()
    .to_dict()
)


Major_Fault downloaded: 75
Major fault features: 75
Geometry types: {'LineString': 69, 'MultiLineString': 6}


## 13. Compute nearest major-fault distance

Malaysia lies close to the equator, so EPSG:3857 introduces relatively small scale distortion for this prototype.

Distance is stored in metres.

This field should still be documented as distance to the **JMG major-fault network exposed by this service**, not distance to every mapped local/minor fault in Malaysia.


In [17]:
DISTANCE_CRS = "EPSG:3857"

sample_m = samples.to_crs(
    DISTANCE_CRS
)

fault_m = faults.to_crs(
    DISTANCE_CRS
)


# Dissolve all major fault lines into one geometry.
fault_union = fault_m.geometry.unary_union


def distance_to_fault(
    point
):

    if (
        point is None
        or point.is_empty
        or fault_union.is_empty
    ):
        return np.nan

    return float(
        point.distance(
            fault_union
        )
    )


sample_m[
    "distance_major_fault_m"
] = sample_m.geometry.apply(
    distance_to_fault
)


samples[
    "distance_major_fault_m"
] = sample_m[
    "distance_major_fault_m"
].to_numpy()


samples[
    "fault_distance_available"
] = samples[
    "distance_major_fault_m"
].notna()


print(
    "Fault distance available:",
    int(
        samples[
            "fault_distance_available"
        ].sum()
    ),
    "/",
    len(samples)
)


display(
    samples[
        [
            "sample_id",
            "distance_major_fault_m"
        ]
    ].head()
)


Fault distance available: 3200 / 3200


C:\Users\User\AppData\Local\Temp\ipykernel_11296\1261131453.py:13: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  fault_union = fault_m.geometry.unary_union


,sample_id,distance_major_fault_m
0,POS_0001,65345.349203
1,POS_0002,62721.198192
2,POS_0003,28.015418
3,POS_0006,8151.388787
4,POS_0007,17024.270826


# PART E — QUALITY CONTROL

## 14. Static-feature validity flags

`core_static_valid` currently requires only:

- valid terrain
- complete four-layer soil coverage

Lithology and fault distance are reported separately rather than automatically rejecting samples, because their nationwide coverage must first be evaluated.

This avoids accidentally discarding large parts of Malaysia merely because an API layer has incomplete coverage.


In [18]:
samples[
    "core_static_valid"
] = (
    samples["terrain_valid"]
    & samples["soil_complete"]
)


samples[
    "all_stage05A_features_available"
] = (
    samples["core_static_valid"]
    & samples["lithology_available"]
    & samples["fault_distance_available"]
)


validity_summary = pd.DataFrame({
    "check": [
        "Total samples",
        "Positive landslides",
        "Background candidates",
        "Terrain valid",
        "Soil complete",
        "Lithology available",
        "Fault distance available",
        "Core static valid",
        "All Stage-05A features available",
    ],
    "value": [
        len(samples),

        int(
            (
                samples[
                    "sample_class"
                ]
                == "landslide"
            ).sum()
        ),

        int(
            (
                samples[
                    "sample_class"
                ]
                == "background_candidate"
            ).sum()
        ),

        int(
            samples[
                "terrain_valid"
            ].sum()
        ),

        int(
            samples[
                "soil_complete"
            ].sum()
        ),

        int(
            samples[
                "lithology_available"
            ].sum()
        ),

        int(
            samples[
                "fault_distance_available"
            ].sum()
        ),

        int(
            samples[
                "core_static_valid"
            ].sum()
        ),

        int(
            samples[
                "all_stage05A_features_available"
            ].sum()
        ),
    ]
})


display(
    validity_summary
)

validity_summary.to_csv(
    OUTPUT_DIR
    / "static_feature_coverage_summary.csv",
    index=False
)


,check,value
0,Total samples,3200
1,Positive landslides,200
2,Background candidates,3000
3,Terrain valid,3200
4,Soil complete,3200
5,Lithology available,3166
6,Fault distance available,3200
7,Core static valid,3200
8,All Stage-05A features available,3166


## 15. Missing-value summary


In [19]:
feature_cols = [
    "elevation_m",
    "slope_deg",
    "aspect_deg",
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
    "lithology_raw",
    "distance_major_fault_m",
]


missing_summary = pd.DataFrame({
    "feature":
        feature_cols,

    "missing_count":
        [
            samples[
                col
            ].isna().sum()
            for col in feature_cols
        ]
})


missing_summary[
    "missing_pct"
] = (
    missing_summary[
        "missing_count"
    ]
    / len(samples)
    * 100
).round(2)


display(
    missing_summary
)

missing_summary.to_csv(
    OUTPUT_DIR
    / "static_feature_missing_summary.csv",
    index=False
)


,feature,missing_count,missing_pct
0,elevation_m,0,0.00
1,slope_deg,0,0.00
2,aspect_deg,0,0.00
3,soil_clay,0,0.00
4,soil_sand,0,0.00
5,soil_silt,0,0.00
6,soil_bulk_density,0,0.00
7,lithology_raw,1849,57.78
8,distance_major_fault_m,0,0.00


## 16. Numerical feature distribution summary

This is only a QC table.

No outlier is removed automatically.


In [20]:
numeric_features = [
    "elevation_m",
    "slope_deg",
    "aspect_deg",
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
    "distance_major_fault_m",
]


distribution_rows = []

for feature in numeric_features:

    values = pd.to_numeric(
        samples[feature],
        errors="coerce"
    )

    distribution_rows.append({
        "feature":
            feature,

        "count":
            int(
                values.notna().sum()
            ),

        "min":
            values.min(),

        "p05":
            values.quantile(0.05),

        "median":
            values.median(),

        "mean":
            values.mean(),

        "p95":
            values.quantile(0.95),

        "max":
            values.max(),
    })


distribution_summary = pd.DataFrame(
    distribution_rows
)


display(
    distribution_summary
)

distribution_summary.to_csv(
    OUTPUT_DIR
    / "static_feature_distribution_summary.csv",
    index=False
)


,feature,count,min,p05,median,mean,p95,max
0,elevation_m,3200,0.00000,8.000000,120.000000,292.235000,1106.150000,3119.000000
1,slope_deg,3200,0.00000,1.000000,9.000000,11.802188,30.000000,60.000000
2,aspect_deg,3200,0.00000,3.950000,174.000000,172.708125,336.050000,358.000000
3,soil_clay,3200,0.00000,0.000000,378.000000,362.997500,445.000000,669.000000
4,soil_sand,3200,0.00000,0.000000,315.000000,301.525625,392.000000,527.000000
5,soil_silt,3200,0.00000,0.000000,300.500000,285.166250,355.000000,438.000000
6,soil_bulk_density,3200,0.00000,0.000000,106.000000,100.194375,118.000000,132.000000
7,distance_major_fault_m,3200,6.26658,2401.542552,26505.043523,34175.754260,90718.528534,148774.292408


## 17. Compare positive vs background feature distributions

This is a **diagnostic only**.

Do not remove background candidates merely because their distributions differ from positives.


In [21]:
class_distribution_rows = []

for feature in numeric_features:

    for sample_class, group in samples.groupby(
        "sample_class"
    ):

        values = pd.to_numeric(
            group[feature],
            errors="coerce"
        )

        class_distribution_rows.append({
            "feature":
                feature,

            "sample_class":
                sample_class,

            "count":
                int(
                    values.notna().sum()
                ),

            "mean":
                values.mean(),

            "median":
                values.median(),

            "std":
                values.std(),

            "min":
                values.min(),

            "max":
                values.max(),
        })


class_distribution = pd.DataFrame(
    class_distribution_rows
)


display(
    class_distribution
)

class_distribution.to_csv(
    OUTPUT_DIR
    / "static_feature_distribution_by_class.csv",
    index=False
)


,feature,sample_class,count,mean,median,std,min,max
0,elevation_m,background_candidate,3000,286.070667,124.000000,350.870506,0.000000,2261.000000
1,elevation_m,landslide,200,384.700000,86.000000,598.024713,0.000000,3119.000000
2,slope_deg,background_candidate,3000,11.775667,9.000000,9.389288,0.000000,60.000000
3,slope_deg,landslide,200,12.200000,8.000000,10.437177,0.000000,42.000000
4,aspect_deg,background_candidate,3000,172.752667,176.000000,106.299163,0.000000,358.000000
5,aspect_deg,landslide,200,172.040000,152.500000,106.703404,0.000000,354.000000
6,soil_clay,background_candidate,3000,372.644000,379.000000,71.121413,0.000000,669.000000
7,soil_clay,landslide,200,218.300000,330.000000,187.277989,0.000000,543.000000
8,soil_sand,background_candidate,3000,308.723667,316.000000,67.343292,0.000000,527.000000
9,soil_sand,landslide,200,193.555000,292.500000,167.240768,0.000000,417.000000


## 18. Save sample-level static feature table

The GeoJSON is convenient for mapping.

The CSV is convenient for QC and later modelling.

These are **not yet the final model-ready datasets**, because:

- background candidates have not yet been reduced to final pseudo-absences
- temporal features are still missing
- soil units/scaling still need confirmation
- lithology categories still need harmonisation


In [22]:
geojson_path = (
    OUTPUT_DIR
    / "static_feature_samples.geojson"
)

csv_path = (
    OUTPUT_DIR
    / "static_feature_samples.csv"
)


export_geo = samples.copy()


# Ensure GeoJSON-safe types
for col in export_geo.columns:

    if pd.api.types.is_datetime64_any_dtype(
        export_geo[col]
    ):

        export_geo[col] = (
            export_geo[col]
            .astype("string")
            .replace("<NA>", None)
        )


export_geo.to_file(
    geojson_path,
    driver="GeoJSON"
)


samples.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    csv_path,
    index=False
)


print("Saved:")
print(geojson_path)

print()
print("Saved:")
print(csv_path)


Saved:
C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features\static_feature_samples.geojson

Saved:
C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features\static_feature_samples.csv


## 19. Save separate valid-background candidate table

This does **not** make them final pseudo-absences.

It only identifies candidates with valid core static data.


In [23]:
valid_background = samples.loc[
    (
        samples[
            "sample_class"
        ]
        == "background_candidate"
    )
    &
    samples[
        "core_static_valid"
    ]
].copy()


valid_bg_geojson = (
    OUTPUT_DIR
    / "background_candidates_core_static_valid.geojson"
)

valid_bg_csv = (
    OUTPUT_DIR
    / "background_candidates_core_static_valid.csv"
)


valid_background.to_file(
    valid_bg_geojson,
    driver="GeoJSON"
)


valid_background.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    valid_bg_csv,
    index=False
)


print(
    "Core-static-valid background candidates:",
    len(valid_background)
)

print(
    "Not yet final pseudo-absences."
)


Core-static-valid background candidates: 3000
Not yet final pseudo-absences.


## 20. Static feature extraction map

This map highlights samples lacking the **core** terrain + soil features.

It does not treat missing lithology as automatically invalid.


In [24]:
fig, ax = plt.subplots(
    figsize=(12, 9)
)


valid = samples[
    samples["core_static_valid"]
]


invalid = samples[
    ~samples["core_static_valid"]
]


valid.plot(
    ax=ax,
    markersize=5,
    alpha=0.35,
    label="Core-static valid"
)


if len(invalid):

    invalid.plot(
        ax=ax,
        marker="x",
        markersize=35,
        label="Terrain/soil missing"
    )


ax.set_title(
    "Stage 05A Static Feature Coverage"
)

ax.set_xlabel(
    "Longitude"
)

ax.set_ylabel(
    "Latitude"
)

ax.legend()

ax.grid(
    True,
    alpha=0.2
)


plt.tight_layout()


map_path = (
    OUTPUT_DIR
    / "static_feature_coverage_map.png"
)


fig.savefig(
    map_path,
    dpi=180,
    bbox_inches="tight"
)

plt.close(fig)


print(
    "Saved:",
    map_path
)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features\static_feature_coverage_map.png


# Stage 05A completion checklist

Before moving to Stage 05B, check:

1. Terrain values exist for nearly all 3,200 samples.
2. Soil values exist across the Malaysia study area.
3. Soil raster units / scale factors are confirmed.
4. Lithology coverage is mapped and missing coverage is understood.
5. JMG major-fault service returned a sensible national network.
6. No samples were discarded simply for having "high-risk" feature values.

---

# Next Stage

## `05B_temporal_feature_extraction.ipynb`

This will handle:

- GLC_FCS30D historical land cover
- Dynamic World recent land cover
- class harmonisation
- historical road distance via ohsome where reliable
- current waterway distance with documented limitation
- GPM rainfall according to date precision

Only after Stage 05B and feature-validity review will we select the final pseudo-absence/background controls.
